# How To & Testing: relational.buildHierarchy
## History
Databricks does not support recursive common table expressions, so we needed to create our own hierarchy builder.

## How it works
The function relataional.buildHierarchy mimics recursive CTE functionality by looping over a given data frame, using the identified child and parent fields to build relationship levels and lineage, returning a data frame with the schema of the source plus two new fields (`level` and `lineage`)

In the event orphans exist (parent not found as a row in source) they are returned with `level = -1`

In the event an infinite loop is detected, remaining rows are returned with `level = -2`.  These can exist for various reasons:
+ A child is its own parent
+ Loop in parent-child relationship (A -> B -> A; X -> Y -> Z -> X; etc.)

In [0]:
dbutils.library.restartPython()

In [0]:
from Tools.Databricks.utils.relational import buildLineage

## Test Data

In [0]:
# Clean hierarchy
df_staff = spark.createDataFrame([
        (1, "Al", "President", None),

        (2, "Betty", "Director", 1),
        (3, "Carl", "Manager", 2),
        (4, "Diane", "Engineer", 3),
        (5, "Evan", "Engineer", 3),

        (6, "Frank", "Director", 1),
        (7, "Gina", "Manager", 6),
        (8, "Harry", "Developer", 7),
        (9, "Irene", "Developer", 7),
        
        (10, "Jack", "Senior Developer", 7),
        (11, "Karen", "Developer", 10)
        ]
    , ["id", "name", "title", "reports_to"])

In [0]:
df_orphans = spark.createDataFrame([
        (12, "Larry", "Engineer", 99),
        (13, "Mary", "Engineer", 101),
        (14, "Nancy", "Engineer", 747)
        ]
    , ["id", "name", "title", "reports_to"])

In [0]:
# 2 Way: O,P,O,P,O,P...
# 3 Way: Q,R,S,Q,R,S...
df_loop = spark.createDataFrame([
        (15, "Oliver", "Engineer", 16),
        (16, "Pamela", "Engineer", 15),
        (17, "Quinn", "Engineer", 19),
        (18, "Ralph", "Engineer", 17),
        (19, "Sara", "Engineer", 18)
        ]
    , ["id", "name", "title", "reports_to"])

In [0]:
# How can one be ones own parent?
df_clones = spark.createDataFrame([
        (20, "Ted", "Engineer", 20),
        (21, "Ursula", "Engineer", 21),
        (22, "Vince", "Engineer", 22)
        ]
    , ["id", "name", "title", "reports_to"])

In [0]:
df_twins = spark.createDataFrame([
        (23, "Wendy", "Sr. Engineer", 7),
        (23, "Wilson", "Sr. Engineer", 7),
        (24, "Xavier", "Engineer", 23),
        (25, "Yvonne", "Engineer", 23),
        (26, "Zachary", "Engineer", 23)
        ]
    , ["id", "name", "title", "reports_to"])

## Testing
### Clean data

In [0]:
# Test with default delimiter and name as token for lineage
df = buildLineage(df_staff, spark, "id", "reports_to", "name")
display(df.sort("level", descending=True).sort("name"))

In [0]:
# Test using Title and a custom delimiter for lineage
df = buildLineage(df_staff, spark, "id", "reports_to", "title", " <- ")
display(df.sort("level", descending=True).sort("name"))

### Sprinkle in some dirty data...

In [0]:
# Test using name and a custom delimiter for lineage
df = buildLineage(df_staff.union(df_orphans), spark, "id", "reports_to", "name", " <- ", True)
display(df.sort("level", descending=True).sort("name"))

In [0]:
# Test using name and a custom delimiter for lineage
# NOTE: Ignore orphans with filter (level >= 0)
df = buildLineage(df_staff.union(df_orphans), spark, "id", "reports_to", "name", " <- ", True)
display(df.sort("level", descending=True).filter("level >= 0").sort("name"))

In [0]:
# display(df_loop)
# Test using name and a custom delimiter for lineage
df = buildLineage(df_staff.union(df_loop), spark, "id", "reports_to", "name", " <- ", True)
display(df.sort("level", descending=True).sort("name"))

In [0]:
# display(df_clones)
# Test using name and a custom delimiter for lineage
df = buildLineage(df_staff.union(df_clones), spark, "id", "reports_to", "name", " <- ", True)
display(df.sort("level", descending=True).sort("name"))

In [0]:
# display(df_twins)
# Test using name and a custom delimiter for lineage
df = buildLineage(df_staff.union(df_twins), spark, "id", "reports_to", "name", " <- ", True)
display(df.sort("level", descending=True).sort("name"))

In [0]:
# Test using name and a custom delimiter for lineage
df = buildLineage(df_staff.union(df_orphans).union(df_loop).union(df_clones).union(df_twins), 
                  spark, 
                  "id", 
                  "reports_to", 
                  "name", 
                  " <- ", 
                  True)
display(df.sort("level", descending=True).sort("name"))